# Set Up pwd and auto updates

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


# Ensure Data Exists 

In [ ]:
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry


get_road_geometry()
get_expected_counts()


# Exparimentation with season functionality

In [ ]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

import seaborn as sns

# Creating Schedules for TM 

In [ ]:
traffic_percentile_schedule = ScheduleSpecs(
     mode='static',
    value=80,  
    dist=None
)

bus_interval_schedule = ScheduleSpecs(
    mode='static',
    value=10,  # static bus interval of 15 minutes
    dist=None
)

crashes_schedule = ScheduleSpecs(
    mode='static',
    value=0,  # static bus interval of 15 minutes
    dist=None # normal distribution for crashes per 100k VMT
)



# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [ ]:
pop_params = PopulationParams(
    population_size=3000,
    prior_car=22.0,
    prior_bus=40.0,
    time_decay_rate=0.1,
    prior_weight=1.0,
    uncertainty_multiplier=1.0,
)


config = make_season_config(
    season_id='speed_test',
    run_description='',
    seed=33,
    n_days=3,
    max_steps=20000,
    max_persons=99999,
    collect_every_n=60,
    start_hr=8,
    bus_capacity=60,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='volume',
    toll_params={"volume_threshold": 100, "slope": 0.05,"base_price": 5.0},   #{'car': 5.0, 'bus': 0.0}, {"volume_threshold": 100, "slope": 0.05,"base_price": 5.0}
    canyon_closures_schedule=None,
    traffic_percentile_schedule=traffic_percentile_schedule,
    bus_interval_schedule=bus_interval_schedule,
    crashes_schedule=crashes_schedule, 
    population_params=pop_params
   
)

In [ ]:
orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
orchestrator.run_season()



In [ ]:
orchestrator.last_model_run

In [ ]:
import cProfile
import pstats

# some stuff used for optimization

def main():
    # Example usage of SeasonOrchestrator with example_config
    orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
    orchestrator.run_season()

if __name__ == "__main__":
    prof = cProfile.Profile()
    prof.enable()

    main()

    prof.disable()
    prof.dump_stats("prof.stats")

    p = pstats.Stats("prof.stats")
    p.strip_dirs().sort_stats("cumulative").print_stats(40)


